In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../')
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import rdkit.Chem as Chem
from rdkit.Chem import Draw
from moldiffusion.chemical import convert_graph_to_smiles, postprocess_smiles, keep_main_molecule
from IPython.display import SVG

## Load Gound Truch and Prediction

In [ ]:
BASE = './data/'
ignore_chiral = False

data = pd.read_csv('./data/real/acs.csv')
pred = pd.read_csv('PATH to prediction.csv')

In [4]:
import sys
sys.path.append('../')
from moldiffusion.chemical import _postprocess_smiles, _convert_graph_to_smiles
from moldiffusion.evaluation import canonicalize_smiles
from moldiffusion.data_aug import CropWhite ,PadToSquare
import albumentations as A
from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.drawOptions.addAtomIndices = True
IPythonConsole.molSize = 600,600

transform = A.Compose([CropWhite(pad=50),PadToSquare(p=1), A.Resize(384,384)])
cnt = 0 
for idx, row in data.iterrows():
    gold_smiles,_ = canonicalize_smiles(row['SMILES'], ignore_cistrans=True, ignore_chiral=ignore_chiral)
    pred_smiles = pred.loc[idx,'SMILES']
    pred_smiles1 = pred.loc[idx,'post_SMILES']
    post_smiles,_ = canonicalize_smiles(pred.loc[idx,'post_SMILES'], ignore_cistrans=True, ignore_chiral=ignore_chiral)
    graph_smiles,_ = canonicalize_smiles(pred.loc[idx,'graph_SMILES'], ignore_cistrans=True, ignore_chiral=ignore_chiral)
    # if gold_smiles == post_smiles:
    #    continue
    print('-' * 20)
    cnt += 1

    print (cnt)
    print(idx)
    print(row['file_path'])
    print('gold:', gold_smiles)
    print('post:', post_smiles)
    print('grph:', graph_smiles)
    print('pred:', pred_smiles)
    print('post1:', pred_smiles1)
    file = str(row['file_path'])
    path = BASE + file
    image = cv2.imread(path)
    print(image.shape)
    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    plt.axis('off')
    plt.imshow(image)
    plt.subplot(1,2,2)
    plt.axis('off')
    img = transform(image=image)['image']
    plt.imshow(img,alpha=0)
    if 'node_coords' in pred.columns:
        coords = np.array(eval(pred.loc[idx, 'node_coords']))
        symbols = eval(pred.loc[idx, 'node_symbols'])
        edges = eval(pred.loc[idx, 'edges'])
        h, w, _ = img.shape
        print(h)
        x, y = coords[:,0]*w, coords[:,1]*h
        plt.scatter(x, y, color='r', marker='o')
        for i in range(len(symbols)):
            plt.text(x[i], y[i], symbols[i], color='blue')
        for i in range(len(x)):
            for j in range(len(x)):
                if edges[i][j] != 0:
                    if edges[i][j] in [5, 6]:
                        color = 'blue' if edges[i][j] == 5 else 'green'#6#虚线
                        plt.arrow(x[i], y[i], x[j]-x[i], y[j]-y[i], color=color, head_width=20)
                    else:
                        color = 'red' if edges[i][j] == 1 else 'orange'
                        plt.plot([x[i], x[j]], [y[i], y[j]], color)
    plt.show()
    mol1 = Chem.MolFromSmiles(gold_smiles)
    mol2 = Chem.MolFromSmiles(post_smiles)
    svg = Draw.MolsToGridImage([mol1, mol2], subImgSize=(250,250), molsPerRow=2, useSVG=True)
    display(svg)
    if cnt == 100:
        break